In [1]:
"""
MLE vs MAP estimation for linear regression.

Derivation:
  Bayes: P(theta|X) ∝ P(X|theta) * P(theta)

  MLE: maximize P(X|theta) only -> ignores the prior.
       Closed form (OLS): theta_hat = (X^T X)^-1 X^T y

  MAP with Gaussian prior theta ~ N(0, tau^2 I):
       log posterior = log P(X|theta) - (1/(2*tau^2)) ||theta||^2
       Maximizing this is exactly Ridge regression:
       theta_hat = (X^T X + lambda*I)^-1 X^T y,  where lambda = sigma^2/tau^2

This file implements both from scratch and verifies the MAP (Ridge) solution
against sklearn.linear_model.Ridge across a regularization path (a range of
lambda values), matching to 6 decimal places.
"""

import numpy as np


def fit_mle(X, y):
    """Closed-form OLS: theta = (X^T X)^-1 X^T y"""
    return np.linalg.solve(X.T @ X, X.T @ y)


def fit_map(X, y, lam):
    """
    Closed-form MAP estimate under a Gaussian prior theta ~ N(0, tau^2 I),
    equivalent to Ridge regression with penalty `lam`.
    theta = (X^T X + lam * I)^-1 X^T y
    Bias term (intercept) is not penalized.
    """
    n_features = X.shape[1]
    I = np.eye(n_features)
    I[0, 0] = 0  # don't penalize the intercept column
    return np.linalg.solve(X.T @ X + lam * I, X.T @ y)


if __name__ == "__main__":
    from sklearn.linear_model import Ridge
    from sklearn.datasets import load_diabetes
    from sklearn.preprocessing import StandardScaler

    # Real dataset (not synthetic): sklearn's diabetes regression dataset —
    # 442 patients, 10 baseline features, target = disease progression.
    data = load_diabetes()
    X_raw, y = data.data, data.target

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    X = np.column_stack([np.ones(len(X_scaled)), X_scaled])  # add intercept

    lambdas = [0.0, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0]

    print(f"{'lambda':>8} | {'max abs diff vs sklearn Ridge':>30}")
    print("-" * 45)
    max_diff_overall = 0.0
    for lam in lambdas:
        theta_map = fit_map(X, y, lam)

        # sklearn Ridge: fit_intercept handled separately, alpha = lam
        ridge = Ridge(alpha=lam, fit_intercept=True)
        ridge.fit(X_scaled, y)
        sklearn_coefs = np.concatenate([[ridge.intercept_], ridge.coef_])

        diff = np.max(np.abs(theta_map - sklearn_coefs))
        max_diff_overall = max(max_diff_overall, diff)
        print(f"{lam:8.1f} | {diff:30.10f}")

    print(f"\nOverall max abs diff across full regularization path: {max_diff_overall:.10f}")
    print("Matches sklearn Ridge to 6 decimal places:", max_diff_overall < 1e-6)

    # MLE == MAP at lambda = 0
    theta_mle = fit_mle(X, y)
    theta_map_0 = fit_map(X, y, 0.0)
    print(f"\nMLE vs MAP(lambda=0) max abs diff: {np.max(np.abs(theta_mle - theta_map_0)):.10f}")


  lambda |  max abs diff vs sklearn Ridge
---------------------------------------------
     0.0 |                   0.0000000000
     0.1 |                   0.0000000000
     1.0 |                   0.0000000000
    10.0 |                   0.0000000000
    50.0 |                   0.0000000000
   100.0 |                   0.0000000000
   500.0 |                   0.0000000000

Overall max abs diff across full regularization path: 0.0000000000
Matches sklearn Ridge to 6 decimal places: True

MLE vs MAP(lambda=0) max abs diff: 0.0000000000
